# EDA

Цель анализа — оценить структуру и качество исследуемого датасета, выявить проблемы: некорректные типы данных, пропуски, некорректные значения, дубликаты и аномалии. Данный анализ необходим для дальнейшей обработки данных.

In [ ]:
import pandas as pd

## 1. Структурная проверка данных

In [ ]:
data = pd.read_csv("../data/finance_ecommerce_dirty_dataset.csv")
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.dtypes

### Вывод по разделу 1

Датасет состоит из 6060 строк и 23 столбцов. Часть полей имеет типы, которые не соответствуют их смыслу: `Date` и `CustomerSince` хранятся как строки, `Amount` и `Balance` имеют строковый тип, а `Phone` и `Postal code` определены как числовые значения.

## 2. Целостность и полнота данных

In [ ]:
data.isna().sum()

In [ ]:
data.isna().mean()

Наибольшая доля пропущенных значений наблюдается в `MerchantEmail` (~30%), `MerchantPhone` (~23%) и `Notes` (~13%). В `CardNumber`, `Phone`, `PostalCode` и `Email` доля пропусков составляет ~4–7%, остальные поля заполнены практически полностью.

In [30]:
data.duplicated().sum()

np.int64(1)

В датасете обнаружен 1 полный дубликат строки.

In [31]:
data["TransactionID"].is_unique

False

In [32]:
data["TransactionID"].dropna().is_unique

False

In [35]:
data["TransactionID"].dropna().duplicated().sum()

np.int64(60)

In [ ]:
data.loc[
    data["TransactionID"].notna()
    & data["TransactionID"].duplicated(keep=False),
    "TransactionID"
].value_counts()

`TransactionID` рассматривался как идентификатор транзакции. В столбце есть 3 пропущенных значения, а среди заполненных значений обнаружено 60 дубликатов идентификаторов. Проверка повторяющихся значений показывает, что такие `TransactionID` встречаются по два раза.

### Вывод по разделу 2

Анализ полноты показал, что данные имеют неодинаковую степень заполнености, определены доли пропусков для каждого типа данных. В датаседе обнаружен один полный дубликат строки. Так же был проведен анализ уникальности идентефикатора, который показал, что при дальнейшей обработке необходимо определить порядок работы с его повторяющимися значениями. 

## 3. Семантический анализ

In [40]:
data["IsFraud"].value_counts(dropna=False)

IsFraud
No      5921
NaN       76
Yess      45
Yes       18
Name: count, dtype: int64

В `IsFraud` встречаются значения `No`, `Yes`, `Yess` и пропуски.

In [41]:
data["CustomerSince"].value_counts(dropna=False)

CustomerSince
unknown       173
2016          154
2025          149
2022          146
2019          144
             ... 
3/25/2020       1
10/18/2022      1
9/5/2025        1
27/11/2017      1
3/9/2025        1
Name: count, Length: 2417, dtype: int64

In [42]:
customer_since_converted = pd.to_datetime(
    data["CustomerSince"],
    errors="coerce",
    format="mixed"
)

data.loc[
    customer_since_converted.isna(),
    "CustomerSince"
].value_counts(dropna=False)

CustomerSince
unknown    173
NaN          2
Name: count, dtype: int64

Столбец `CustomerSince` содержит даты в нескольких форматах, а также значение `unknown`. При преобразовании в datetime с использованием `format="mixed"` большинство значений распознаются корректно. Не преобразуются 173 значения `unknown`, 2 значения являются пропусками.

In [47]:
data["Date"].head(20)

0       4/3/2025
1      10/8/2024
2      9/14/2024
3     12/21/2024
4      4/14/2024
5      5/20/2025
6      9/14/2023
7     10/23/2023
8     10/31/2023
9       2/7/2024
10      7/8/2024
11    10/13/2024
12     11/1/2023
13     2/24/2025
14      5/3/2024
15     8/25/2024
16     9/12/2024
17     6/20/2025
18    10/10/2024
19     5/10/2024
Name: Date, dtype: str

In [48]:
date_converted = pd.to_datetime(
    data["Date"],
    errors="coerce",
    format="%m/%d/%Y"
)

data.loc[
    date_converted.isna(),
    "Date"
].value_counts(dropna=False)

Series([], Name: count, dtype: int64)

Столбец `Date` хранится в строковом формате, однако все значения соответствуют формату месяц/день/год (%m/%d/%Y). При преобразовании в datetime некорректных значений не обнаружено. Следовательно, проблема столбца заключается только в исходном типе данных, а дополнительная очистка значений не требуется.

In [ ]:

amount_numeric = pd.to_numeric(data["Amount"], errors="coerce")

data.loc[
    amount_numeric.isna(),
    "Amount"
].value_counts(dropna=False).head(10)

Amount
2.50x        3
$73.66       2
33.16x       2
INR 14.68    2
Rs 63.57     2
24.92x       2
INR 16.83    2
INR 6.62     2
USD 27.19    2
USD 28.08    2
Name: count, dtype: int64

In [58]:
balance_numeric = pd.to_numeric(data["Balance"], errors="coerce")

data.loc[
    balance_numeric.isna(),
    "Balance"
].value_counts(dropna=False).head(10)

Balance
NaN               72
?-554,639.12       2
?-595,131.99       2
?-1,714,295.08     2
?-176,002.59       2
?-175,307.66       2
?-182,672.26       2
?-1,617,269.64     2
?-2,773,860.42     2
?-1,849,660.56     2
Name: count, dtype: int64

В `Amount` и `Balance` встречаются значения, которые нельзя напрямую преобразовать в числовой тип. Поэтому при приведении типов используется `pd.to_numeric(..., errors="coerce")`: некорректные значения становятся пропусками. Такие значения не могут быть напрямую преобразованы в числовой тип и требуют отдельной обработки.

In [59]:
data["ExchangeRate"].describe()

count    6058.000000
mean       46.319508
std        40.254595
min         1.000000
25%         1.000000
50%        72.091800
75%        80.274425
max       109.968400
Name: ExchangeRate, dtype: float64

`ExchangeRate` уже имеет числовой тип. Минимальное значение равно 1, максимальное — около 110,отрицательных и нулевых значений не обнаружено. Явных семантических аномалий в столбце не выявлено.

In [61]:
data["MerchantPhone"].value_counts(dropna=False).head(15)

MerchantPhone
 NaN             1413
 9.590231e+08       6
-7.122014e+08       5
 5.669108e+09       5
 1.689022e+09       5
 6.655838e+07       5
 3.635479e+09       5
 9.473747e+09       5
-1.529532e+09       5
-7.080457e+09       5
 2.144366e+09       5
 3.991187e+09       4
-5.256654e+09       4
 8.595112e+08       4
-7.940931e+08       4
Name: count, dtype: int64

In [60]:
data["PostalCode"].value_counts(dropna=False).head(15)

PostalCode
NaN         318
851.0         3
880938.0      2
292.0         2
366839.0      2
767751.0      2
460.0         2
676348.0      2
331559.0      2
958281.0      2
221228.0      2
791338.0      2
905570.0      2
733.0         2
202809.0      2
Name: count, dtype: int64

In [62]:
data["Phone"].value_counts(dropna=False).head(15)

Phone
 NaN             328
 8.885465e+09      2
 9.669522e+08      2
 1.051015e+09      2
 5.840069e+09      2
 9.167525e+09      2
-9.262595e+09      2
 5.628702e+09      2
 6.495085e+09      2
 1.599957e+09      2
 3.604458e+08      2
-2.129881e+09      2
 8.618752e+09      2
 3.703928e+09      2
 5.488630e+09      2
Name: count, dtype: int64


`MerchantPhone`, `PostalCode` и `Phone` используются как идентификаторы, их логичнее хранить как строки, а не как числовые значения. Из-за числового типа значения отображаются как числа с плавающей точкой или в экспоненциальной форме. В телефонных полях также встречаются отрицательные значения, которые не соответствуют смыслу признака и являются аномальными. 

### Вывод по разделу 3

Проверка показала несколько проблем: некорректные категории, разные форматы дат, лишние символы в числовых значениях и неподходящие типы данных для некоторых идентификаторов.

При этом `Date` и `ExchangeRate` не содержат явных проблем и требуют минимальной обработки.